# ML-05 — Feature Vector and Leakage/Privacy Check

## 1. Build the feature vector

This continues the ML-04 data contract on the real **warehouse** (`hf://datasets/FlyRank/internship-warehouse`), not the starter CSV. I set an **anchor of 2026-03-31** and split time cleanly:

- **Feature window** = the trailing 90 days before the anchor, read as three 30-day blocks: **March** (`prev`), **February** (`mid`), **January** (`old`). Everything here is history, known at the anchor.
- **Label window** = the 30 days *after* the anchor, **April 2026** — a genuine forward window. `future_decline = April impressions < 0.8 × March impressions`. June 2026 (`_sample`) stays sealed, exactly as the contract said.

There is **no window overlap**: features stop at 2026-03-31, the label starts 2026-04-01. The query aggregates the four monthly partitions once and caches the result to `work/outputs/feature_vector.csv`; ML-06→ML-10 read that cache and never re-scan the warehouse (per the dataset's scan-once rule to avoid 429s).

The token is read from the environment / a local `.env` / Colab Secrets — never pasted in a cell (public repo).

In [1]:
import os
from pathlib import Path
import numpy as np, pandas as pd, duckdb

SEED = 42
ANCHOR = "2026-03-31"
BASE = "hf://datasets/FlyRank/internship-warehouse"
OUT = Path("../outputs"); OUT.mkdir(parents=True, exist_ok=True)

def hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    env = Path("../../.env")
    if env.exists():
        for line in env.read_text().splitlines():
            if line.strip().startswith("HF_TOKEN") and "=" in line:
                return line.split("=", 1)[1].strip().strip('"').strip("'")
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        from getpass import getpass
        return getpass("HF_TOKEN: ")

con = duckdb.connect()
con.execute("INSTALL httpfs;"); con.execute("LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{hf_token()}')")
print("connected to warehouse; anchor =", ANCHOR)

connected to warehouse; anchor = 2026-03-31


In [2]:
FACT = "[" + ",".join(f"'{BASE}/fact_content_daily_performance/month=2026-0{i}/*.parquet'"
                       for i in [1, 2, 3, 4]) + "]"

sql = f"""
WITH elig AS (
  SELECT client_hash_id FROM read_parquet('{BASE}/dim_clients.parquet')
  WHERE has_gsc_access AND gsc_data_start <= DATE '2026-01-01'
),
perf AS (
  SELECT client_hash_id, content_hash_id,
    SUM(gsc_impressions) FILTER (WHERE month='2026-03')                          AS impr_prev,
    SUM(gsc_clicks)      FILTER (WHERE month='2026-03')                          AS clicks_prev,
    SUM(ga4_sessions)    FILTER (WHERE month='2026-03' AND ga4_data_available)   AS sess_prev,
    SUM(gsc_impressions) FILTER (WHERE month='2026-02')                          AS impr_mid,
    SUM(gsc_impressions) FILTER (WHERE month='2026-01')                          AS impr_old,
    SUM(gsc_impressions) FILTER (WHERE month='2026-04')                          AS impr_future
  FROM read_parquet({FACT}, hive_partitioning=true, union_by_name=true)
  WHERE client_hash_id IN (SELECT client_hash_id FROM elig)
  GROUP BY 1, 2
)
SELECT p.client_hash_id, p.content_hash_id,
       p.impr_prev, p.clicks_prev, p.sess_prev, p.impr_mid, p.impr_old, p.impr_future,
       c.content_type, c.main_intent, c.competition_level,
       c.search_volume, c.competition, c.cpc, c.word_count, c.char_count,
       c.content_created_date, c.content_updated_date
FROM perf p
JOIN read_parquet('{BASE}/dim_content.parquet') c USING (client_hash_id, content_hash_id)
WHERE p.impr_prev > 0
  AND c.is_published AND NOT c.is_deleted
  AND c.content_created_date <= DATE '{ANCHOR}'
"""
raw = con.sql(sql).df()
print("pairs fetched from warehouse:", f"{len(raw):,}", "| clients:", raw.client_hash_id.nunique())

pairs fetched from warehouse: 145,030 | clients: 28


In [3]:
anchor = pd.Timestamp(ANCHOR)
for c in ["impr_prev", "clicks_prev", "sess_prev", "impr_mid", "impr_old", "impr_future"]:
    raw[c] = raw[c].fillna(0.0)

raw["future_decline"] = (raw["impr_future"] < 0.8 * raw["impr_prev"]).astype(int)
print("decline base rate:", round(raw["future_decline"].mean(), 3))

eps = 1.0
raw["ctr_prev_30d"]        = raw["clicks_prev"] / (raw["impr_prev"] + eps)
raw["hist_impr_momentum"]  = (raw["impr_prev"] - raw["impr_mid"]) / (raw["impr_mid"] + eps)
raw["log_impressions_prev_30d"] = np.log1p(raw["impr_prev"])
raw["log_clicks_prev_30d"]      = np.log1p(raw["clicks_prev"])
raw["log_sessions_prev_30d"]    = np.log1p(raw["sess_prev"])
raw["log_impr_older_30d"]       = np.log1p(raw["impr_old"])
raw["log_search_volume"]        = np.log1p(raw["search_volume"].fillna(0))

created = pd.to_datetime(raw["content_created_date"])
updated = pd.to_datetime(raw["content_updated_date"])
eff_update = updated.where(updated.notna() & (updated <= anchor), created)
raw["content_age_days"]       = (anchor - created).dt.days.clip(lower=0)
raw["days_since_last_update"] = (anchor - eff_update).dt.days.clip(lower=0)
raw["age_tier_order"] = pd.cut(raw["content_age_days"], [-1, 7, 30, 90, 180, 365, 1e9], labels=False) + 1

raw["has_keyword_data"] = raw["search_volume"].notna().astype(int)
raw["has_word_count"]   = raw["word_count"].notna().astype(int)
raw["reach_impr"] = raw["impr_prev"].round().astype(int)
raw["content_id"] = raw["content_hash_id"]
raw["client_id"]  = raw["client_hash_id"]

NUM = ["log_impressions_prev_30d","log_clicks_prev_30d","log_sessions_prev_30d","log_impr_older_30d",
       "ctr_prev_30d","hist_impr_momentum","log_search_volume","competition","cpc",
       "word_count","char_count","content_age_days","days_since_last_update","age_tier_order",
       "has_keyword_data","has_word_count"]
CAT = ["content_type","main_intent","competition_level"]
for c in NUM:
    raw[c] = pd.to_numeric(raw[c], errors="coerce").fillna(0.0)
for c in CAT:
    raw[c] = raw[c].fillna("unknown").astype(str)

feat = raw[["content_id","client_id"] + NUM + CAT + ["future_decline","reach_impr"]].copy()
feat = feat.sort_values("content_id").reset_index(drop=True)
feat.to_csv(OUT / "feature_vector.csv", index=False)
print(f"wrote {OUT/'feature_vector.csv'}  ->  {feat.shape[0]:,} rows x {feat.shape[1]} cols")
print("features:", len(NUM), "numeric +", len(CAT), "categorical")

decline base rate: 0.544


wrote ../outputs/feature_vector.csv  ->  145,030 rows x 23 cols
features: 16 numeric + 3 categorical


## 2. Feature notes (meaning, missing, categorical, available-when?)

Every feature is aggregated from the **trailing 90 days** (Jan–Mar 2026) or from static `dim_content` attributes — all knowable at the 2026-03-31 anchor, before the April label window. Missingness is handled with explicit `has_*` flags rather than a blind `fillna(0)`, which would smuggle a content-type signal in.

| Feature | Meaning | Missing handling | Available before the decision? |
|---|---|---|---|
| `log_impressions_prev_30d` | log1p March GSC impressions | zero-filled | Yes — history |
| `log_clicks_prev_30d` | log1p March GSC clicks | zero-filled | Yes — history |
| `log_sessions_prev_30d` | log1p March GA4 sessions (only where `ga4_data_available`) | zero-filled | Yes — history |
| `log_impr_older_30d` | log1p January impressions (deepest block) | zero-filled | Yes — history |
| `ctr_prev_30d` | March clicks / March impressions | +1 smoothing | Yes — historical rate |
| `hist_impr_momentum` | (March − Feb) / Feb impressions — a **past→past** trajectory | +1 smoothing | Yes — no future in it |
| `log_search_volume` | log1p keyword search volume | `has_keyword_data` flag + 0 | Yes — static |
| `competition`, `cpc` | keyword competition / cost-per-click | 0 when absent | Yes — static |
| `word_count`, `char_count` | article size | `has_word_count` flag + 0 | Yes — static |
| `content_age_days` | anchor − `content_created_date` | none | Yes — static at anchor |
| `days_since_last_update` | anchor − last update **on or before** the anchor | falls back to created date | Yes — future updates excluded |
| `age_tier_order` | ordinal age bucket 1–6 | none | Yes — derived from age |
| `content_type`, `main_intent`, `competition_level` | categorical context | `"unknown"` | Yes — static |

Note `days_since_last_update` deliberately ignores any `content_updated_date` **after** the anchor — using a future refresh would leak. The cell below shows keyword/word-count missingness tracks `content_type`, which is why the `has_*` flags exist.

In [4]:
miss = (raw.assign(kw_missing=raw["has_keyword_data"].eq(0),
                   wc_missing=raw["has_word_count"].eq(0))
           .groupby("content_type")[["kw_missing", "wc_missing"]].mean().round(3))
miss.columns = ["% keyword-data missing", "% word_count missing"]
print(miss.to_string())

                    % keyword-data missing  % word_count missing
content_type                                                    
comparison article                   0.000                 0.002
feedly article                       1.000                 0.000
keyword article                      0.018                 0.421


## 3. The leakage hunt

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def make_pipe(num_cols):
    pre = ColumnTransformer([("num", StandardScaler(), num_cols),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, random_state=SEED))])

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED).split(raw, groups=raw["client_id"]))
y_tr, y_te = raw["future_decline"].iloc[tr], raw["future_decline"].iloc[te]

auc_honest = roc_auc_score(y_te, make_pipe(NUM).fit(raw.iloc[tr][NUM + CAT], y_tr)
                           .predict_proba(raw.iloc[te][NUM + CAT])[:, 1])

raw["impr_future_leak"] = np.log1p(raw["impr_future"])
NUM_LEAK = NUM + ["impr_future_leak"]
auc_leak = roc_auc_score(y_te, make_pipe(NUM_LEAK).fit(raw.iloc[tr][NUM_LEAK + CAT], y_tr)
                         .predict_proba(raw.iloc[te][NUM_LEAK + CAT])[:, 1])

print(f"test base rate (share declining): {y_te.mean():.3f}")
print(f"honest grouped-split AUC (history + static only): {auc_honest:.3f}")
print(f"LEAKY AUC (April impressions added):              {auc_leak:.3f}   <- the tell")

test base rate (share declining): 0.388
honest grouped-split AUC (history + static only): 0.557
LEAKY AUC (April impressions added):              1.000   <- the tell


**Verdict — the harness works, and the honest signal is modest.** Feeding the model the April label window (`impr_future_leak`) sends AUC toward 1.0: one feature towering over the rest plus a near-perfect score is the textbook leakage tell, and seeing it jump confirms the test can actually catch a leak. Removed, the honest grouped-split AUC lands only a little above the base rate — a page's own trailing trajectory plus static properties carries **weak, directional** signal about the next month, not a crystal ball. That small number is the real one. This is why the lane is framed as decision-support triage, not prediction of Google.

## 4. What I excluded and why

Fields I refused to feed the model, each with one line of why:

| Excluded field(s) | Why it is out |
|---|---|
| April 2026 (`impr_future`, any label-window day) | **The future** — this *is* what I'm predicting. |
| `gsc_avg_position`, `gsc_sum_position` (over the label window) | Would only be a feature if taken strictly from the trailing window; the forward slice is out. |
| `content_updated_date` **after** the anchor | A refresh that happened *after* the decision moment — future information. |
| The FlyRank Health Score / optimization flags | **Decision-derived** product flags — using them means learning the old rule, not the world. They may be a baseline to beat, never a feature. |
| `fact_content_query_90d` | Its fixed 90-day window is not aligned to my per-anchor windows and overlaps the label months — excluded until alignment is built and checked. |
| `provider_used`, `model_used` | Dictionary marks these not model features. |
| `content_hash_id`, `client_hash_id` | Pseudonymous keys — for grouped splits and joins only, never features. |

The cell below asserts none of these entered the saved matrix.

In [6]:
FORBIDDEN = {"impr_future","impr_future_leak","gsc_avg_position","gsc_sum_position",
             "content_updated_date","health_score","provider_used","model_used",
             "content_hash_id","client_hash_id"}
saved = pd.read_csv(OUT / "feature_vector.csv")
leaked = FORBIDDEN & set(saved.columns)
assert not leaked, f"LEAK: forbidden columns in feature_vector.csv -> {leaked}"
meta_cols = {'content_id','client_id','future_decline','reach_impr'}
print("guard passed: no forbidden column in feature_vector.csv")
print(f"final matrix: {saved.shape[0]:,} rows, {len(set(saved.columns)-meta_cols)} model features")
print("reach_impr is carried for downstream value-gating, not a model feature")
print("features:", [c for c in saved.columns if c not in meta_cols])

guard passed: no forbidden column in feature_vector.csv
final matrix: 145,030 rows, 19 model features
reach_impr is carried for downstream value-gating, not a model feature
features: ['log_impressions_prev_30d', 'log_clicks_prev_30d', 'log_sessions_prev_30d', 'log_impr_older_30d', 'ctr_prev_30d', 'hist_impr_momentum', 'log_search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'age_tier_order', 'has_keyword_data', 'has_word_count', 'content_type', 'main_intent', 'competition_level']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.